In [1]:
import pandas as pd
import sqlite3
import io
from datetime import datetime

# ==============================================================================
# 1. CARGA Y GENERACIÓN DE DATASETS ORIGINALES (CSV Sucios)
# ==============================================================================
estudiantes_raw = """id_estudiante,estudiante_info,email,edad
1,Alessandro Di Mauro - Premium,ale@correo.com,22
2,Gabriela Ruiz - Basico,,21
3,Maria Jose Huertas - Premium,majo@correo.com,
4,carlos perez - basico,cperez@empresa.co,25"""

profesores_raw = """id_profesor,profesor_info,departamento
10,Diego Zuluaga - Tecnologia,Ingenieria
11,Ana Gomez - Matematicas,Ciencias"""

cursos_raw = """id_curso,curso_info,precio,cupos,id_profesor
101,Programacion Python - Basico,150000,30,10
102,Bases de Datos - Intermedio,200000,25,10
103,Calculo Aplicado - Avanzado,180000,20,11"""

# Generamos los archivos físicos sucios
with open('estudiantes_sucios.csv', 'w') as f: f.write(estudiantes_raw)
with open('profesores_sucios.csv', 'w') as f: f.write(profesores_raw)
with open('cursos_sucios.csv', 'w') as f: f.write(cursos_raw)

# ==============================================================================
# 2. PANDAS Y NORMALIZACIÓN (1NF)
# ==============================================================================
# Estudiantes
df_est = pd.read_csv(io.StringIO(estudiantes_raw))
df_est[['nombre', 'suscripcion']] = df_est['estudiante_info'].str.split(' - ', expand=True)
df_est['nombre'] = df_est['nombre'].str.title()
df_est['email'] = df_est['email'].fillna('sin_correo@edutech.com')
df_est['edad'] = df_est['edad'].fillna(18) # Imputación básica
df_est = df_est.drop(columns=['estudiante_info'])

# Profesores
df_prof = pd.read_csv(io.StringIO(profesores_raw))
df_prof[['nombre', 'especialidad']] = df_prof['profesor_info'].str.split(' - ', expand=True)
df_prof = df_prof.drop(columns=['profesor_info'])

# Cursos
df_cur = pd.read_csv(io.StringIO(cursos_raw))
df_cur[['nombre_curso', 'nivel']] = df_cur['curso_info'].str.split(' - ', expand=True)
df_cur = df_cur.drop(columns=['curso_info'])

# Exportar limpios
df_est.to_csv('estudiantes_limpios.csv', index=False)
df_prof.to_csv('profesores_limpios.csv', index=False)
df_cur.to_csv('cursos_limpios.csv', index=False)

# ==============================================================================
# 3. MIGRACIÓN SQLITE Y ARQUITECTURA
# ==============================================================================
conn = sqlite3.connect('parcial_edutech.db')
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

df_est.to_sql('estudiantes', conn, if_exists='replace', index=False)
df_prof.to_sql('profesores', conn, if_exists='replace', index=False)
df_cur.to_sql('cursos', conn, if_exists='replace', index=False)

# Relación N:M (Inscripciones equivale a Ventas)
cursor.execute('''
CREATE TABLE IF NOT EXISTS inscripciones (
    id_inscripcion INTEGER PRIMARY KEY AUTOINCREMENT,
    id_estudiante INTEGER,
    id_curso INTEGER,
    monto_pagado REAL,
    fecha TEXT,
    FOREIGN KEY (id_estudiante) REFERENCES estudiantes(id_estudiante),
    FOREIGN KEY (id_curso) REFERENCES cursos(id_curso)
)''')
conn.commit()

# ==============================================================================
# 4. BONO: POO Y CRUD
# ==============================================================================
class PlataformaEduTech:
    def __init__(self, db): self.db = db

    def crear_inscripcion(self, id_est, id_cur):
        with sqlite3.connect(self.db) as c:
            precio = c.execute("SELECT precio FROM cursos WHERE id_curso = ?", (id_cur,)).fetchone()[0]
            fecha = datetime.now().strftime('%Y-%m-%d')
            c.execute("INSERT INTO inscripciones (id_estudiante, id_curso, monto_pagado, fecha) VALUES (?,?,?,?)",
                           (id_est, id_cur, precio, fecha))
            print(f"✅ [CREATE] Inscripción exitosa.")

    def leer_reporte_join(self):
        with sqlite3.connect(self.db) as c:
            query = '''
            SELECT i.id_inscripcion, e.nombre, c.nombre_curso, p.nombre AS profesor, i.monto_pagado
            FROM inscripciones i
            JOIN estudiantes e ON i.id_estudiante = e.id_estudiante
            JOIN cursos c ON i.id_curso = c.id_curso
            JOIN profesores p ON c.id_profesor = p.id_profesor
            '''
            print("\n--- REPORTE DE INSCRIPCIONES (JOIN) ---")
            print(pd.read_sql(query, c))

    def actualizar_cupos(self, id_cur, nuevos_cupos):
        with sqlite3.connect(self.db) as c:
            c.execute("UPDATE cursos SET cupos = ? WHERE id_curso = ?", (nuevos_cupos, id_cur))
            print("✅ [UPDATE] Cupos actualizados.")

    def eliminar_inscripcion(self, id_ins):
        with sqlite3.connect(self.db) as c:
            c.execute("DELETE FROM inscripciones WHERE id_inscripcion = ?", (id_ins,))
            print("✅ [DELETE] Inscripción cancelada.")

# Pruebas
app = PlataformaEduTech('parcial_edutech.db')
app.crear_inscripcion(1, 101)
app.crear_inscripcion(2, 102)
app.crear_inscripcion(3, 103)
app.leer_reporte_join()
app.actualizar_cupos(101, 29)
app.eliminar_inscripcion(2)
conn.close()

✅ [CREATE] Inscripción exitosa.
✅ [CREATE] Inscripción exitosa.
✅ [CREATE] Inscripción exitosa.

--- REPORTE DE INSCRIPCIONES (JOIN) ---
   id_inscripcion               nombre         nombre_curso       profesor  \
0               1  Alessandro Di Mauro  Programacion Python  Diego Zuluaga   
1               2        Gabriela Ruiz       Bases de Datos  Diego Zuluaga   
2               3   Maria Jose Huertas     Calculo Aplicado      Ana Gomez   

   monto_pagado  
0      150000.0  
1      200000.0  
2      180000.0  
✅ [UPDATE] Cupos actualizados.
✅ [DELETE] Inscripción cancelada.
